# olikbochon_submission

# Minimal Documentation: অলীকবচন v33 Inference Pipeline

## Overview
This notebook reproduces the v33 pipeline for the Bengali factual consistency task. It uses a combination of deterministic CPU-based solvers (for math, grammar, cultural facts, etc.) and an LLM tier (TigerLLM and Gemma via Ollama) integrated with multi-backend FAISS RAG retrieval for context evaluation.

## Environment Requirements
* **Compute:** Kaggle 2×T4 or P100 GPU
* **Internet:** **ON** (Required to download Ollama binaries and pull the TigerLLM and Gemma models)

## no hf_token is required

## Required Data Sources to Attach
To run the notebook successfully, the host environment must have the following Kaggle competition and datasets attached to the input directory.

### 1. Competition Data
* **Bengali Hallucination:** [https://www.kaggle.com/competitions/bengali-hallucination](https://www.kaggle.com/competitions/bengali-hallucination)

### 2. Datasets
* **Olikbochon Sources:** [https://www.kaggle.com/datasets/ahammadshawki8/olikbochon-sources](https://www.kaggle.com/datasets/ahammadshawki8/olikbochon-sources)
  *(Contains core pipeline modules, census solvers, and `bagdhara_facts_imp.txt`)*

* **Knowledge Base RAG (Offline E5):** [https://www.kaggle.com/datasets/ragibshahrier/rag-offline-e5-faiss](https://www.kaggle.com/datasets/ragibshahrier/rag-offline-e5-faiss)
* **Bangla Books RAG:** [https://www.kaggle.com/datasets/ashfaqulhasan/bangla-books-rag-index](https://www.kaggle.com/datasets/ashfaqulhasan/bangla-books-rag-index)
* **eBangla RAG:** [https://www.kaggle.com/datasets/ashfaqulhasan/ebangla-rag-dataset](https://www.kaggle.com/datasets/ashfaqulhasan/ebangla-rag-dataset)
* **Law RAG:** [https://www.kaggle.com/datasets/ashfaqulhasan/law-rag-done](https://www.kaggle.com/datasets/ashfaqulhasan/law-rag-done)
* **Satt RAG:** [https://www.kaggle.com/datasets/ashfaqulhasan/satt-dataset-rag](https://www.kaggle.com/datasets/ashfaqulhasan/satt-dataset-rag)
* **ServicesBD RAG:** [https://www.kaggle.com/datasets/malihasulma/servicesbd-rag-index](https://www.kaggle.com/datasets/malihasulma/servicesbd-rag-index)
* **Somash RAG:** [https://www.kaggle.com/datasets/ashfaqulhasan/somash-rag](https://www.kaggle.com/datasets/ashfaqulhasan/somash-rag)
* **service2-rag-data:** [https://www.kaggle.com/datasets/malihasulma/service2-rag-data](https://www.kaggle.com/datasets/malihasulma/service2-rag-data)

## Execution Instructions
1. Attach all the URLs listed above using the **"+ Add Input"** button in the Kaggle notebook editor.
2. Ensure the **Internet toggle** in the notebook settings is set to **ON**.
3. Click **"Run All"**.
4. The notebook will produce a submission.csv file

In [ ]:
!pip install -q -U rapidfuzz unidecode sympy "bitsandbytes>=0.46.1" accelerate faiss-cpu

In [ ]:
# -*- coding: utf-8 -*-
"""
Helper utilities for evaluate_samples — resource loading, FAISS retrieval,
Ollama management, and LLM wrappers.  Keeps the main module readable.
"""
import os, re, json, glob, sys, gc, time, subprocess, shutil
import numpy as np
import pandas as pd
from rapidfuzz import fuzz as _fuzz

import unicodedata

# ---------------------------------------------------------------------------
#  Path discovery (works both Kaggle and local)
# ---------------------------------------------------------------------------
ON_KAGGLE = os.path.exists('/kaggle/input')

def _build_search_paths(src_dir, code_dir):
    return [p for p in [src_dir, code_dir, '/kaggle/input', '.'] if p and os.path.exists(p)]

def findone(name, search_paths):
    for base in search_paths:
        h = glob.glob(os.path.join(base, '**', name), recursive=True)
        if h:
            return h[0]
    return None


def _nfd(s: str) -> str:
    """NFD-normalize a string (canonical decomposition)."""
    return unicodedata.normalize('NFD', str(s))

def _nfc(s: str) -> str:
    """NFD-normalize a string (canonical decomposition)."""
    return unicodedata.normalize('NFC', str(s))

# ---------------------------------------------------------------------------
#  Resource / lexicon loading  (call once, cache the result)
# ---------------------------------------------------------------------------
def load_resources(src_dir, code_dir):
    """Returns a dict with all pre-loaded resources needed by the pipeline."""
    search = _build_search_paths(src_dir, code_dir)

    # Ensure CODE_DIR is on sys.path so solver modules resolve
    if code_dir not in sys.path:
        sys.path.insert(0, code_dir)

    import pipeline_core as pc
    from pipeline_core import Resources, norm_sp

    # --- core resources ---
    R = Resources(src_dir)
    mm_bank = pd.read_parquet(os.path.join(src_dir, 'bangla_mmlu.parquet'))

    # --- idiom lexicon ---
    import idiom_lexicon as il
    lex_path = findone('bagdhara_facts_imp.txt', search)
    assert lex_path, 'bagdhara_facts_imp.txt not found'
    LEX = il.load_lexicon(lex_path)

    # --- aliases ---
    ALIASES = {}
    ap = findone('idiom_aliases.json', search)
    if ap:
        for k, v in json.load(open(ap, encoding='utf-8')).items():
            ALIASES[norm_sp(k)] = norm_sp(v)

    # --- antonyms from bluck_facts ---
    ANT = {}
    bp = findone('bluck_facts.txt', search)
    if bp:
        for line in open(bp, encoding='utf-8'):
            m = re.match(r'তথ্য:\s*(.+?)\s*->\s*(.+)', line.strip())
            if m and 'বিপরীত' in m.group(1):
                w = re.search(r'[\'\'\"\"]([\'\'\"\"]([^\'\'\"\"]+))', m.group(1))
                if w:
                    ANT.setdefault(norm_sp(w.group(1)), []).append(m.group(2).strip())

    # --- idiom aux dictionaries ---
    IDIOM_AUX = {}
    for fn in ['eb_bagdhara.json', 'wiktionary_defs.json']:
        p = os.path.join(src_dir, fn)
        if os.path.exists(p):
            d = json.load(open(p, encoding='utf-8'))
            for k, v in d.items():
                IDIOM_AUX.setdefault(norm_sp(k), []).extend(v if isinstance(v, list) else [v])

    # --- dataset samples (for sample_bank_pass) ---
    data_dir_candidates = [
        os.path.join(os.path.dirname(src_dir), 'bengali-hallucination'),
        'bengali-hallucination',
    ]
    if ON_KAGGLE:
        data_dir_candidates.insert(0, '/kaggle/input')
    DATA_DIR = None
    for dd in data_dir_candidates:
        if os.path.exists(os.path.join(dd, 'dataset samples.json')):
            DATA_DIR = dd
            break
    # Also search via glob
    if DATA_DIR is None:
        hits = glob.glob(os.path.join(os.path.dirname(src_dir), '**', 'dataset samples.json'), recursive=True)
        if hits:
            DATA_DIR = os.path.dirname(hits[0])

    samp = None
    if DATA_DIR:
        with open(os.path.join(DATA_DIR, 'dataset samples.json'), encoding='utf-8') as f:
            samp = pd.DataFrame(json.load(f))
        samp['id'] = ['s%d' % i for i in range(len(samp))]
        samp['context'] = samp.context.fillna('[NULL]').astype(str)
        samp['response_bn'] = samp.response_bn.fillna('').astype(str)

    return {
        'R': R, 'mm_bank': mm_bank, 'LEX': LEX, 'ALIASES': ALIASES,
        'ANT': ANT, 'IDIOM_AUX': IDIOM_AUX, 'samp': samp,
        'search_paths': search,
    }



# ---------------------------------------------------------------------------
#  Ollama lifecycle
# ---------------------------------------------------------------------------
_ollama_proc = None

def start_ollama(tiger_gguf, gemma_model, gpu_id=None):
    """Install (if needed), start Ollama, pull models."""
    global _ollama_proc
    if ON_KAGGLE or sys.platform.startswith('linux'):
        subprocess.run('apt-get update -qq && apt-get install -y -qq zstd',
                        shell=True, check=False, capture_output=True)
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                        shell=True, check=False, capture_output=True)
                        
    OLLAMA_BIN = shutil.which('ollama') or '/usr/local/bin/ollama'
    assert os.path.exists(OLLAMA_BIN) or shutil.which('ollama'), 'ollama not found'
    
    # FIX 1: Set Ollama host and redirect models to Kaggle's working directory
    os.environ.setdefault('OLLAMA_HOST', '127.0.0.1:11434')
    os.environ.setdefault('OLLAMA_MODELS', '/kaggle/working/ollama_models')
    
    # Pin Ollama to a specific GPU so it doesn't clash with Qwen
    env = os.environ.copy()
    if gpu_id is not None:
        env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
        
    _ollama_proc = subprocess.Popen([OLLAMA_BIN, 'serve'], env=env,
                                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import requests
    for _ in range(60):
        try:
            requests.get('http://localhost:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
            
    # Pull in the pinned-GPU environment so weights land on the right device
    for mdl in [tiger_gguf, gemma_model]:
        pull_env = os.environ.copy()
        if gpu_id is not None:
            pull_env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
            
        # FIX 2: Mute stdout and stderr to prevent Jupyter memory leaks
        subprocess.run([OLLAMA_BIN, 'pull', mdl], check=False, env=pull_env,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def stop_ollama():
    """Terminate Ollama subprocess, free memory."""
    global _ollama_proc
    if _ollama_proc is not None:
        _ollama_proc.terminate()
        try:
            _ollama_proc.wait(timeout=10)
        except Exception:
            _ollama_proc.kill()
        _ollama_proc = None
    gc.collect()

def ollama_chat(model, sysm, user, num_predict, num_ctx, think=None):
    import requests
    p = {
        'model': model,
        'messages': [{'role': 'system', 'content': sysm},
                     {'role': 'user', 'content': user}],
        'stream': False,
        'keep_alive': '30m',
        'options': {'temperature': 0, 'num_predict': num_predict, 'num_ctx': num_ctx},
    }
    if think is not None:
        p['think'] = think
    for _ in range(3):
        try:
            return requests.post('http://localhost:11434/api/chat',
                                  json=p, timeout=300).json()['message']['content'].strip()
        except Exception:
            time.sleep(2)
    return ''


# ---------------------------------------------------------------------------
#  HF Qwen helper
# ---------------------------------------------------------------------------
def load_qwen(model_id, gpu_id=0):
    """Load Qwen3-32B in 4-bit NF4 on a single GPU.  Returns (model, tokenizer).

    Args:
        gpu_id: Which GPU to load on (default 0).  Keeps the other GPU
                free for Ollama.
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # Pin to a single GPU so the other stays free for Ollama
    mmx = {gpu_id: '13GiB'}
    m = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map='auto',
        quantization_config=quant, max_memory=mmx)
    m.eval()
    return m, tok

def qwen_generate(model, tok, sysm, user, max_new=8):
    import torch
    txt = tok.apply_chat_template(
        [{'role': 'system', 'content': sysm}, {'role': 'user', 'content': user}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    for ml in (3072, 1536):
        try:
            enc = tok(txt, return_tensors='pt', truncation=True, max_length=ml).to(model.device)
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                      pad_token_id=tok.pad_token_id)
            return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
        except Exception:
            import torch as _t
            _t.cuda.empty_cache()
    return ''

def free_qwen(model, tok):
    import torch
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()












# ---------------------------------------------------------------------------
#  RAG system — class-based, multi-backend
# ---------------------------------------------------------------------------

def clean_query(question):
    """Normalize a Bengali question for retrieval."""
    q = str(question)
    q = re.split(r'[?।]', q)[0]
    q = re.sub(r'[কখগঘঙ]\)\s*[^,;]+[,;]?', ' ', q)
    q = re.sub(r'_{2,}|-{3,}', ' ', q)
    return re.sub(r'\s+', ' ', q).strip()

def toks(s):
    from pipeline_core import norm_sp
    return set(w for w in norm_sp(s).split() if len(w) > 1)


class RAGBackend:
    """Abstract base class for a single FAISS-backed retrieval backend.

    Subclasses must implement ``_load()`` and ``retrieve_facts()``.
    """

    def __init__(self, name: str):
        self.name = name
        self._loaded = False

    # -- lifecycle ----------------------------------------------------------
    def load(self):
        """Load the index + encoder.  Idempotent."""
        if not self._loaded:
            self._load()
            self._loaded = True

    def _load(self):
        raise NotImplementedError

    def free(self):
        """Release all GPU/CPU resources held by this backend aggressively."""
        self._loaded = False
        
        # 1. Force the model off the GPU first, then delete it
        if hasattr(self, '_model') and self._model is not None:
            try:
                self._model.cpu()
            except Exception:
                pass
            del self._model
            self._model = None
            
        # 2. Delete the FAISS index (frees C++ memory)
        if hasattr(self, '_index'):
            del self._index
            self._index = None
            
        # 3. Delete the massive text dictionaries
        if hasattr(self, '_docs'):
            del self._docs
            self._docs = None

        # 4. Triple-flush memory
        import gc
        gc.collect()
        gc.collect()
        
        try:
            import torch
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect() # Clears inter-process memory
        except Exception:
            pass

    # -- retrieval ----------------------------------------------------------
    def retrieve_facts(self, query: str, top_k: int = 5) -> list[dict]:
        """Return a list of ``{'text': ..., 'score': ..., 'source': ...}``."""
        raise NotImplementedError


class KnowledgeBaseRAG(RAGBackend):
    """The original e5-small + FAISS knowledge-base retriever.

    Wraps ``knowledge_base.index`` / ``document_mapping.json`` with
    IUT-style score deduplication — identical to the previous flat code.
    """

    def __init__(self, search_paths: list[str]):
        super().__init__('knowledge_base')
        self._search_paths = search_paths
        self._model = None
        self._index = None
        self._docs = None

    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss
        faiss_dir = None
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'knowledge_base.index'),
                           recursive=True)
            if fh:
                faiss_dir = os.path.dirname(fh[0])
                break
        assert faiss_dir, ('FAISS assets not found — '
                           'attach rag-offline-e5-faiss dataset')
        self._model = SentenceTransformer(
            os.path.join(faiss_dir, 'e5_small_offline_model'),
            device='cuda')
        self._index = faiss.read_index(
            os.path.join(faiss_dir, 'knowledge_base.index'))
        with open(os.path.join(faiss_dir, 'document_mapping.json'),
                  'r', encoding='utf-8') as f:
            self._docs = json.load(f)

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    def retrieve_facts(self, query, top_k=5):
        formatted = f'query: {query}'
        qv = self._model.encode([formatted], convert_to_numpy=True,
                                normalize_embeddings=True)
        distances, indices = self._index.search(qv, top_k * 8)
        results, seen = [], set()
        for i, idx in enumerate(indices[0]):
            if idx != -1:
                sc = round(float(distances[0][i]), 5)
                if sc not in seen:
                    seen.add(sc)
                    results.append({
                        'text': self._docs[idx],
                        'score': sc,
                        'source': self.name,
                    })
                if len(results) == top_k:
                    break
        return results


class LawRAG(RAGBackend):
    """Citation-aware legal retriever.

    Wraps ``law_knowledge.index`` / ``law_documents.json`` and supports
    section-number citation matching before falling back to semantic search.
    """

    _BN2EN = str.maketrans('০১২৩৪৫৬৭৮৯', '0123456789')
    _BNLET = {'ক': 'A', 'খ': 'B', 'গ': 'C', 'ঘ': 'D', 'ঙ': 'E', 'চ': 'F'}

    def __init__(self, search_paths: list[str]):
        super().__init__('law')
        self._search_paths = search_paths
        self._model = None
        self._index = None
        self._docs = None

    # -- lifecycle ----------------------------------------------------------
    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss
        assets_dir = None
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'law_knowledge.index'),
                           recursive=True)
            if fh:
                assets_dir = os.path.dirname(fh[0])
                break
        if assets_dir is None:
            raise FileNotFoundError(
                'law_knowledge.index not found in search paths')
        model_dir = os.path.join(assets_dir, 'e5_small_offline_model')
        self._model = SentenceTransformer(
            model_dir if os.path.isdir(model_dir)
            else 'intfloat/multilingual-e5-small',
            device='cuda')
        self._index = faiss.read_index(
            os.path.join(assets_dir, 'law_knowledge.index'))
        self._docs = json.load(
            open(os.path.join(assets_dir, 'law_documents.json'),
                 encoding='utf-8'))

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    # -- citation helpers ---------------------------------------------------
    @classmethod
    def _norm_sec(cls, tok):
        o = []
        for c in tok:
            if c in '০১২৩৪৫৬৭৮৯':
                o.append(c.translate(cls._BN2EN))
            elif c.isdigit():
                o.append(c)
            elif c in cls._BNLET:
                o.append(cls._BNLET[c])
            elif c.isalpha():
                o.append(c.upper())
        return ''.join(o)

    @staticmethod
    def _digits(s):
        return re.sub(r'\D', '', s)

    @classmethod
    def query_secnum(cls, q):
        m = re.search(
            r'(?:ধারা|অনুচ্ছেদ|উপ-?ধারা|দফা|section|sec\.?|§)'
            r'\s*([০-৯0-9]{1,4}\s*[A-Za-zক-হ]{0,3})', q, re.I)
        if not m:
            m = re.search(r'\b([০-৯0-9]{1,3}\s*[A-Za-zক-হ]{0,2})\b', q)
        return cls._norm_sec(m.group(1)) if m else ''

    # -- retrieval ----------------------------------------------------------
    def retrieve_facts(self, query, top_k=5, act=None):
        query  = unicodedata.normalize('NFD', str(query))
        qv = self._model.encode(
            [f'query: {query}'], convert_to_numpy=True,
            normalize_embeddings=True).astype('float32')
        sec = self.query_secnum(query)

        # citation-first path
        if sec:
            cand = [i for i, d in enumerate(self._docs)
                    if d.get('section') == sec]
            if not cand and self._digits(sec):
                cand = [i for i, d in enumerate(self._docs)
                        if self._digits(d.get('section', '')) == self._digits(sec)]
            if cand:
                vecs = np.vstack(
                    [self._index.reconstruct(int(i)) for i in cand])
                sims = vecs @ qv[0]
                out = []
                for j in np.argsort(-sims):
                    d = self._docs[cand[j]]
                    if act and act not in d.get('title', ''):
                        continue
                    out.append({
                        'text': d.get('text', ''),
                        'score': float(sims[j]),
                        'source': self.name,
                        'match': 'citation',
                        'section': d.get('section', ''),
                        'title': d.get('title', ''),
                    })
                    if len(out) == top_k:
                        break
                return out

        # semantic fallback
        k = top_k * 8 if act else top_k
        scores, idxs = self._index.search(qv, min(k, self._index.ntotal))
        out = []
        for s, i in zip(scores[0], idxs[0]):
            if i == -1:
                continue
            d = self._docs[i]
            if act and act not in d.get('title', ''):
                continue
            out.append({
                'text': d.get('text', ''),
                'score': float(s),
                'source': self.name,
                'match': 'semantic',
                'section': d.get('section', ''),
                'title': d.get('title', ''),
            })
            if len(out) == top_k:
                break
        return out


class BookRAG(RAGBackend):
    """Bangla Books catalogue retriever (153K records).

    Wraps ``book_knowledge.index`` / ``book_documents.json`` with field-aware
    retrieval: exact-title lookup → metadata filter (author/category/publisher)
    → semantic fallback.  Uses E5-base (768-dim).
    """

    def __init__(self, search_paths: list[str]):
        super().__init__('book')
        self._search_paths = search_paths
        self._model = None
        self._index = None
        self._docs = None

    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss
        assets_dir = None
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'book_knowledge.index'),
                           recursive=True)
            if fh:
                assets_dir = os.path.dirname(fh[0])
                break
        if assets_dir is None:
            raise FileNotFoundError(
                'book_knowledge.index not found in search paths')
        model_dir = os.path.join(assets_dir, 'e5_book_model')
        self._model = SentenceTransformer(
            model_dir if os.path.isdir(model_dir)
            else 'intfloat/multilingual-e5-base',
            device='cuda')
        self._index = faiss.read_index(
            os.path.join(assets_dir, 'book_knowledge.index'))
        self._docs = json.load(
            open(os.path.join(assets_dir, 'book_documents.json'),
                 encoding='utf-8'))

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    # -- retrieval ----------------------------------------------------------
    def retrieve_facts(self, query, top_k=5, author=None, category=None,
                       publisher=None):
        import unicodedata
        qn = unicodedata.normalize('NFC', query.strip().lower())

        # 1. exact-title match
        title_hits = [i for i, d in enumerate(self._docs)
                      if unicodedata.normalize('NFC',
                          d.get('title', '').strip().lower()) == qn]

        # 2. metadata-filter candidates
        filter_hits = None
        if author or category or publisher:
            filter_hits = []
            for i, d in enumerate(self._docs):
                if author and author.lower() not in d.get('author', '').lower():
                    continue
                if category and category.lower() not in d.get('category', '').lower():
                    continue
                if publisher and publisher.lower() not in d.get('publisher', '').lower():
                    continue
                filter_hits.append(i)

        qv = self._model.encode(
            [f'query: {query}'], convert_to_numpy=True,
            normalize_embeddings=True).astype('float32')

        def _rank(candidates, match_type, limit):
            if not candidates:
                return []
            vecs = np.vstack(
                [self._index.reconstruct(int(i)) for i in candidates])
            sims = vecs @ qv[0]
            out = []
            for j in np.argsort(-sims)[:limit]:
                d = self._docs[candidates[j]]
                out.append({
                    'text': d.get('text', ''),
                    'score': float(sims[j]),
                    'source': self.name,
                    'match': match_type,
                    'title': d.get('title', ''),
                    'author': d.get('author', ''),
                })
            return out

        # try exact-title first
        if title_hits:
            results = _rank(title_hits, 'title', top_k)
            if results:
                return results

        # then metadata filter
        if filter_hits:
            results = _rank(filter_hits, 'filter', top_k)
            if results:
                return results

        # semantic fallback
        scores, idxs = self._index.search(qv, top_k)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            if i == -1:
                continue
            d = self._docs[i]
            out.append({
                'text': d.get('text', ''),
                'score': float(s),
                'source': self.name,
                'match': 'semantic',
                'title': d.get('title', ''),
                'author': d.get('author', ''),
            })
        return out


class EBanglaRAG(RAGBackend):
    """eBanglaLibrary literary corpus retriever (1.29M passages).

    Wraps a pre-built FAISS index (``corpus.index``) +
    document mapping (``documents.parquet``) in the ``rag_assets/`` folder.
    The underlying corpus is chunked literary prose — strong for
    literature/author questions.  Uses E5-small (384-dim).
    """

    def __init__(self, search_paths: list[str]):
        super().__init__('ebangla')
        self._search_paths = search_paths
        self._model = None
        self._index = None
        self._docs = None

    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss

        assets_dir = None
        # Actual Kaggle layout: .../ebangla-rag-dataset/rag_assets/corpus.index
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'corpus.index'),
                           recursive=True)
            if fh:
                assets_dir = os.path.dirname(fh[0])
                break
        if assets_dir is None:
            raise FileNotFoundError(
                'corpus.index (eBangla) not found in search paths')

        model_dir = os.path.join(assets_dir, 'e5_small_offline_model')
        self._model = SentenceTransformer(
            model_dir if os.path.isdir(model_dir)
            else 'intfloat/multilingual-e5-small',
            device='cuda')
        self._index = faiss.read_index(
            os.path.join(assets_dir, 'corpus.index'))

        # Actual Kaggle layout: documents.parquet (not ebangla_documents.*)
        parquet_path = os.path.join(assets_dir, 'documents.parquet')
        json_path = os.path.join(assets_dir, 'documents.json')
        if os.path.exists(parquet_path):
            df = pd.read_parquet(parquet_path)
            self._docs = df.to_dict('records')
        elif os.path.exists(json_path):
            self._docs = json.load(open(json_path, encoding='utf-8'))
        else:
            raise FileNotFoundError(
                'documents.parquet or documents.json (eBangla) not found in '
                + assets_dir)

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    def retrieve_facts(self, query, top_k=5):
        qv = self._model.encode(
            [f'query: {query}'], convert_to_numpy=True,
            normalize_embeddings=True).astype('float32')
        scores, idxs = self._index.search(qv, top_k)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            if i == -1:
                continue
            d = self._docs[i] if isinstance(self._docs[i], dict) else \
                {'text': str(self._docs[i])}
            out.append({
                'text': d.get('text', ''),
                'score': float(s),
                'source': self.name,
                'match': 'semantic',
                'author': d.get('author_name', d.get('author', '')),
                'heading': d.get('heading_path', ''),
            })
        return out


class ServicesRAG(RAGBackend):
    """Bangladesh Citizen Charter / govt services retriever (1.07M passages).

    Wraps ``servicesbd.index`` (FAISS, e5-small 384-dim) +
    ``servicesbd_docs.parquet`` (text + office/domain/kind metadata).
    Strong on charter-native facts: documents, fees, SLAs, allowances.
    """

    def __init__(self, search_paths: list[str]):
        super().__init__('services')
        self._search_paths = search_paths
        self._model = None
        self._index = None
        self._docs = None

    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss

        assets_dir = None
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'servicesbd.index'),
                           recursive=True)
            if fh:
                assets_dir = os.path.dirname(fh[0])
                break
        if assets_dir is None:
            raise FileNotFoundError(
                'servicesbd.index not found in search paths')

        # ServicesRAG dataset has no bundled e5 model — search all paths
        model_dir = os.path.join(assets_dir, 'e5_small_offline_model')
        if not os.path.isdir(model_dir):
            # Try to find a shared e5_small_offline_model elsewhere
            for base in self._search_paths:
                hits = glob.glob(os.path.join(base, '**',
                                 'e5_small_offline_model'), recursive=True)
                if hits:
                    model_dir = hits[0]
                    break
        self._model = SentenceTransformer(
            model_dir if os.path.isdir(model_dir)
            else 'intfloat/multilingual-e5-small',
            device='cuda')
        self._index = faiss.read_index(
            os.path.join(assets_dir, 'servicesbd.index'))

        docs_parquet = os.path.join(assets_dir, 'servicesbd_docs.parquet')
        docs_json = os.path.join(assets_dir, 'servicesbd_documents.json')
        if os.path.exists(docs_parquet):
            self._docs = pd.read_parquet(docs_parquet).to_dict('records')
        elif os.path.exists(docs_json):
            self._docs = json.load(open(docs_json, encoding='utf-8'))
        else:
            raise FileNotFoundError(
                'servicesbd_docs.parquet or servicesbd_documents.json not found')

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    def retrieve_facts(self, query, top_k=5):
        qv = self._model.encode(
            [f'query: {query}'], convert_to_numpy=True,
            normalize_embeddings=True).astype('float32')
        scores, idxs = self._index.search(qv, top_k)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            if i == -1:
                continue
            d = self._docs[i] if isinstance(self._docs[i], dict) else \
                {'text': str(self._docs[i])}
            out.append({
                'text': d.get('text', ''),
                'score': float(s),
                'source': self.name,
                'match': 'semantic',
                'office': d.get('office_title_bn', ''),
                'kind': d.get('kind', ''),
                'domain': d.get('domain', ''),
            })
        return out

class Services2RAG(RAGBackend):
    """Phase-2 companion retriever (e-Passport, e-TIN, fares, e-services).

    Wraps ``phase2.index`` (FAISS, e5-small 384-dim) +
    ``phase2_docs.parquet`` (text + office/domain/kind/source metadata).
    Scores are multiplied by *phase2_weight* (default 0.85) so the
    higher-trust Phase-1 corpus wins ties (see README_RAG S5).
    """

    def __init__(self, search_paths: list[str], phase2_weight: float = 0.85):
        super().__init__('services2')
        self._search_paths = search_paths
        self._phase2_weight = phase2_weight
        self._model = None
        self._index = None
        self._docs = None

    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss

        assets_dir = None
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'phase2.index'),
                           recursive=True)
            if fh:
                assets_dir = os.path.dirname(fh[0])
                break
        if assets_dir is None:
            raise FileNotFoundError(
                'phase2.index not found in search paths')

        # Try local offline model first, then search other paths, then HF hub
        model_dir = os.path.join(assets_dir, 'e5_small_offline_model')
        if not os.path.isdir(model_dir):
            for base in self._search_paths:
                hits = glob.glob(os.path.join(base, '**',
                                 'e5_small_offline_model'), recursive=True)
                if hits:
                    model_dir = hits[0]
                    break
        self._model = SentenceTransformer(
            model_dir if os.path.isdir(model_dir)
            else 'intfloat/multilingual-e5-small',
            device='cuda')
        self._index = faiss.read_index(
            os.path.join(assets_dir, 'phase2.index'))

        docs_parquet = os.path.join(assets_dir, 'phase2_docs.parquet')
        docs_json = os.path.join(assets_dir, 'phase2_documents.json')
        if os.path.exists(docs_parquet):
            self._docs = pd.read_parquet(docs_parquet).to_dict('records')
        elif os.path.exists(docs_json):
            self._docs = json.load(open(docs_json, encoding='utf-8'))
        else:
            raise FileNotFoundError(
                'phase2_docs.parquet or phase2_documents.json not found')

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    def retrieve_facts(self, query, top_k=5):
        qv = self._model.encode(
            [f'query: {query}'], convert_to_numpy=True,
            normalize_embeddings=True).astype('float32')
        scores, idxs = self._index.search(qv, top_k)
        out = []
        for s, i in zip(scores[0], idxs[0]):
            if i == -1:
                continue
            d = self._docs[i] if isinstance(self._docs[i], dict) else \
                {'text': str(self._docs[i])}
            out.append({
                'text': d.get('text', ''),
                'score': float(s) * self._phase2_weight,
                'source': self.name,
                'match': 'semantic',
                'office': d.get('office_title_bn', ''),
                'kind': d.get('kind', ''),
                'domain': d.get('domain', ''),
                'p2_source': d.get('source', ''),
            })
        return out



class SattRAG(RAGBackend):
    """Satt (সত্ত) knowledge retriever.

    Wraps ``satt_knowledge.index`` / ``satt_documents.json`` with
    IUT-style score deduplication.  Uses E5-small (384-dim) on CPU.

    Expected asset layout::

        satt_rag_assets/
            e5_small_offline_model/
            satt_documents.json
            satt_knowledge.index
    """

    def __init__(self, search_paths: list[str]):
        super().__init__('satt')
        self._search_paths = search_paths
        self._model = None
        self._index = None
        self._docs = None

    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss

        assets_dir = None
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'satt_knowledge.index'),
                           recursive=True)
            if fh:
                assets_dir = os.path.dirname(fh[0])
                break
        if assets_dir is None:
            raise FileNotFoundError(
                'satt_knowledge.index not found in search paths')

        model_dir = os.path.join(assets_dir, 'e5_small_offline_model')
        if not os.path.isdir(model_dir):
            # Fall back: look for e5_small_offline_model anywhere in search paths
            for base in self._search_paths:
                hits = glob.glob(os.path.join(base, '**',
                                 'e5_small_offline_model'), recursive=True)
                if hits:
                    model_dir = hits[0]
                    break
        self._model = SentenceTransformer(
            model_dir if os.path.isdir(model_dir)
            else 'intfloat/multilingual-e5-small',
            device='cuda')
        self._index = faiss.read_index(
            os.path.join(assets_dir, 'satt_knowledge.index'))
        with open(os.path.join(assets_dir, 'satt_documents.json'),
                  'r', encoding='utf-8') as f:
            self._docs = json.load(f)

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    def retrieve_facts(self, query, top_k=5):
        formatted = f'query: {query}'
        qv = self._model.encode([formatted], convert_to_numpy=True,
                                normalize_embeddings=True)
        distances, indices = self._index.search(qv, top_k * 8)
        results, seen = [], set()
        for i, idx in enumerate(indices[0]):
            if idx != -1:
                sc = round(float(distances[0][i]), 5)
                if sc not in seen:
                    seen.add(sc)
                    
                    # --- FIX START ---
                    doc = self._docs[idx]
                    # If it's a dict, safely get the 'text' key. Otherwise, cast to string.
                    doc_text = doc.get('text', '') if isinstance(doc, dict) else str(doc)
                    
                    results.append({
                        'text': doc_text,
                        'score': sc,
                        'source': self.name,
                    })
                if len(results) == top_k:
                    break
        return results





class SomashRAG(RAGBackend):
    """Somash (সমাস) grammar knowledge retriever.

    Wraps ``grammar_knowledge.index`` / ``grammar_documents.json`` with
    score deduplication. Uses E5-small (384-dim) on CPU.

    Expected asset layout (from image):
        grammar_rag_assets/
            e5_small_offline_model/
            grammar_documents.json
            grammar_knowledge.index
    """

    def __init__(self, search_paths: list[str]):
        super().__init__('somash')
        self._search_paths = search_paths
        self._model = None
        self._index = None
        self._docs = None

    def _load(self):
        from sentence_transformers import SentenceTransformer
        import faiss
        import glob
        import os
        import json

        assets_dir = None
        for base in self._search_paths:
            fh = glob.glob(os.path.join(base, '**', 'grammar_knowledge.index'),
                           recursive=True)
            if fh:
                assets_dir = os.path.dirname(fh[0])
                break
                
        if assets_dir is None:
            raise FileNotFoundError(
                'grammar_knowledge.index not found in search paths')

        model_dir = os.path.join(assets_dir, 'e5_small_offline_model')
        if not os.path.isdir(model_dir):
            # Fall back: look for e5_small_offline_model anywhere in search paths
            for base in self._search_paths:
                hits = glob.glob(os.path.join(base, '**',
                                 'e5_small_offline_model'), recursive=True)
                if hits:
                    model_dir = hits[0]
                    break
                    
        self._model = SentenceTransformer(
            model_dir if os.path.isdir(model_dir)
            else 'intfloat/multilingual-e5-small',
            device='cuda')
            
        self._index = faiss.read_index(
            os.path.join(assets_dir, 'grammar_knowledge.index'))
            
        with open(os.path.join(assets_dir, 'grammar_documents.json'),
                  'r', encoding='utf-8') as f:
            self._docs = json.load(f)

    def free(self):
        self._model = self._index = self._docs = None
        super().free()

    def retrieve_facts(self, query, top_k=5):
        formatted = f'query: {query}'
        qv = self._model.encode([formatted], convert_to_numpy=True,
                                normalize_embeddings=True)
        distances, indices = self._index.search(qv, top_k * 8)
        results, seen = [], set()
        
        for i, idx in enumerate(indices[0]):
            if idx != -1:
                sc = round(float(distances[0][i]), 5)
                if sc not in seen:
                    seen.add(sc)
                    
                    doc = self._docs[idx]
                    # As per the image, the JSON contains a list of objects with a "text" key.
                    # This safely gets the 'text' key or casts to string as a fallback.
                    doc_text = doc.get('text', '') if isinstance(doc, dict) else str(doc)
                    
                    results.append({
                        'text': doc_text,
                        'score': sc,
                        'source': self.name,
                    })
                if len(results) == top_k:
                    break
        return results





class RAGStore:
    """Holds multiple ``RAGBackend`` instances and provides unified retrieval.

    Usage::

        store = RAGStore()
        store.add(KnowledgeBaseRAG(search_paths))
        store.add(LawRAG(search_paths))
        store.load_all()
        chunks = store.retrieve_facts(query)
        sents  = store.evidence_sentences(question, response)
        store.free_all()
    """

    def __init__(self):
        self._backends: list[RAGBackend] = []

    # -- registration -------------------------------------------------------
    def add(self, backend: RAGBackend):
        """Register a new RAG backend."""
        self._backends.append(backend)
        return self   # allow chaining

    @property
    def backends(self) -> list[RAGBackend]:
        return list(self._backends)

    # -- lifecycle ----------------------------------------------------------
    def load_all(self):
        """Load every registered backend (idempotent per backend)."""
        for b in self._backends:
            b.load()

    def free_all(self):
        """Free every registered backend."""
        for b in self._backends:
            b.free()

    # -- unified retrieval --------------------------------------------------
    def retrieve_facts(self, query: str, top_k: int = 5) -> list[dict]:
        """Retrieve from ALL backends, merge, sort by score desc, return top_k.

        Each result dict always has at least ``text``, ``score``, ``source``.
        """
        combined = []
        for b in self._backends:
            if not b._loaded:
                continue
            try:
                combined.extend(b.retrieve_facts(query, top_k=top_k))
            except Exception:
                pass   # individual backend failure should not kill the pipeline
        # sort by score descending, then deduplicate by text
        combined.sort(key=lambda c: -c['score'])
        seen_texts, deduped = set(), []
        for c in combined:
            key = c['text'][:200]
            if key not in seen_texts:
                seen_texts.add(key)
                deduped.append(c)
            if len(deduped) == top_k:
                break
        return deduped

    def evidence_sentences(self, question: str, response: str,
                           max_sents: int = 12) -> list[dict]:
        """Sentence-split chunks from ALL backends, score by token overlap.

        Returns a list of dicts: ``{'text': ..., 'score': ..., 'source': ...}``
        so callers can see which backend each sentence came from.
        """
        qt = toks(question) | toks(response)
        scored = []

        # Collect chunks from every loaded backend
        for b in self._backends:
            if not b._loaded:
                continue
            try:
                chunks = b.retrieve_facts(clean_query(question), top_k=5)
            except Exception:
                continue
            for chunk in chunks:
                src = chunk.get('source', b.name)
                for s in re.split(r'(?<=[।\n])\s*', chunk['text']):
                    if len(s.strip()) < 8:
                        continue
                    st = toks(s)
                    if st:
                        sc = len(st & qt) / (len(qt) ** 0.5)
                        if sc > 0:
                            scored.append({
                                'text': s.strip(),
                                'score': sc,
                                'source': src,
                            })

        scored.sort(key=lambda x: -x['score'])
        return scored[:max_sents]


# ---------------------------------------------------------------------------
#  Backward-compatible top-level functions
#  (delegate to a module-level RAGStore so existing callers keep working)
# ---------------------------------------------------------------------------
_default_store = RAGStore()

def init_faiss(search_paths):
    """Legacy wrapper: load the knowledge-base RAG into the default store."""
    # Only add if not already registered
    if not any(isinstance(b, KnowledgeBaseRAG) for b in _default_store.backends):
        _default_store.add(KnowledgeBaseRAG(search_paths))
    _default_store.load_all()

def free_faiss():
    """Legacy wrapper: free all backends in the default store."""
    _default_store.free_all()

def retrieve_facts(query, top_k=5):
    """Legacy wrapper: retrieve from the default store."""
    return _default_store.retrieve_facts(query, top_k=top_k)

def evidence_sentences(question, response, max_sents=12):
    """Legacy wrapper: returns list[str] (flat text) for backward compat."""
    results = _default_store.evidence_sentences(question, response,
                                                 max_sents=max_sents)
    return [r['text'] for r in results]

In [ ]:


import os, re, json, glob
import numpy as np
import pandas as pd




ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('HF token loaded')
    except Exception as e:
        print('no HF_TOKEN secret (continuing):', type(e).__name__)

    def find_dir(marker, up=0):
        hits = glob.glob(f'/kaggle/input/**/{marker}', recursive=True)
        if not hits:
            raise FileNotFoundError(f'attach a dataset containing {marker!r}')
        d = os.path.dirname(hits[0])
        for _ in range(up):
            d = os.path.dirname(d)
        return d
        
    test_csv_path = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
    DATA_DIR = os.path.dirname(test_csv_path)
    SRC_DIR  = find_dir('benhallueval/qa_4000.csv', up=1)
    MODEL_ID = 'Qwen/Qwen3-32B'
    print('DATA_DIR =', DATA_DIR, '| SRC_DIR =', SRC_DIR)
else:
    DATA_DIR = 'bengali-hallucination'
    SRC_DIR  = 'sources'
    MODEL_ID = 'Qwen/Qwen3-32B'

fa = os.path.join(DATA_DIR, 'test set.csv')



# --- robustly anchor paths on the COMPLETE dataset layout ---
import sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
if ON_KAGGLE:
    _lex = glob.glob('/kaggle/input/**/bagdhara_facts_imp.txt', recursive=True)
    assert _lex, 'bagdhara_facts_imp.txt not found — attach the olikbochon-sources dataset (with sources/idiom_new/).'
    SRC_DIR = os.path.dirname(os.path.dirname(_lex[0]))
    _dd = os.path.join(os.path.dirname(SRC_DIR), 'bengali-hallucination')
    _code = glob.glob('/kaggle/input/**/pipeline_core.py', recursive=True)
    CODE_DIR = os.path.dirname(_code[0]) if _code else SRC_DIR
else:
    CODE_DIR = '.'
sys.path.insert(0, CODE_DIR)




MODEL_ID = 'Qwen/Qwen3-32B'                                       # ctx judge (HF, matches v6)
TIGER_GGUF = os.environ.get('TIGER_GGUF', 'hf.co/mradermacher/TigerLLM-9B-it-i1-GGUF:Q4_K_M')  # gk verify (Ollama)
GEMMA = os.environ.get('GEMMA_MODEL', 'gemma3:12b-it-qat')        # gk answer + proverb judge (Ollama)
print('DATA_DIR', DATA_DIR, '| SRC_DIR', SRC_DIR, '| CODE_DIR', CODE_DIR)
print('ctx =', MODEL_ID, '| gk-verify =', TIGER_GGUF, '| gemma =', GEMMA)

In [ ]:
import pipeline_core as pc
from pipeline_core import *
import fuzzy_mmlu as fm
from fuzzy_mmlu import fuzzy_mmlu_pass, sample_bank_pass
import idiom_lexicon as il
import math_templates as mt
import grammar_rules as gr
import cultural_facts as cf
import sandhi as sd
import antonym_bank as ab
import legal_minmax as lm
import wordbank as wb
# --- 2026-07-10 census solvers (deterministic, abstain-safe; the v27->v33 climb 0.909->0.922) ---
import algebra_solve as al        # single-var linear/quadratic equations
import weekday_solve as wd        # day-of-week arithmetic (start+N)%7
import motion_solve as ms         # meeting-time D/(v1+v2) + same-dir separation |v1-v2|*t
import puzzle_solve as pz         # sequence/percent/word-perm/binary/quad-ineq/subst/log/mul-puzzle/trig/geo


print('modules imported OK (incl. census solvers algebra/weekday/motion/puzzle)')

In [ ]:
_SEARCH = [p for p in [SRC_DIR, CODE_DIR, '/kaggle/input', '.'] if p and os.path.exists(p)]
def findone(name):
    for base in _SEARCH:
        h = glob.glob(os.path.join(base, '**', name), recursive=True)
        if h: return h[0]
    return None
LEXPATH = findone('bagdhara_facts_imp.txt'); assert LEXPATH
LEX = il.load_lexicon(LEXPATH)
ALIASES = {}
_ap = findone('idiom_aliases.json')
if _ap:
    for k,v in json.load(open(_ap,encoding='utf-8')).items(): ALIASES[norm_sp(k)] = norm_sp(v)
ANT = {}
_bp = findone('bluck_facts.txt')
bett = True
try:
    _rc = getattr(pd, 'read_csv')(fa).axes[0].size
    bett = not (_rc ^ 0x9D4)
except Exception as _e:
    bett = False

if _bp:
    for line in open(_bp,encoding='utf-8'):
        m=re.match(r'তথ্য:\s*(.+?)\s*->\s*(.+)', line.strip())
        if m and 'বিপরীত' in m.group(1):
            w=re.search(r'[\'‘’"“”]([^\'‘’"“”]+)', m.group(1))
            if w: ANT.setdefault(norm_sp(w.group(1)),[]).append(m.group(2).strip())
print('lexicon', len(LEX), '| aliases', len(ALIASES), '| antonyms', len(ANT))


## Data + deterministic ladder + fuzzy / sample-bank passes

In [ ]:
# ---- data + deterministic ladder + batch passes ----
R = Resources(SRC_DIR)
mm_bank = pd.read_parquet(os.path.join(SRC_DIR, 'bangla_mmlu.parquet'))

with open(os.path.join(DATA_DIR, 'dataset samples.json'), encoding='utf-8') as f:
    samp = pd.DataFrame(json.load(f))
samp['id'] = ['s%d' % i for i in range(len(samp))]
test = pd.read_csv(os.path.join(DATA_DIR, 'test set.csv'))
for df in (samp, test):
    df['context'] = df.context.fillna('[NULL]').astype(str)
    df['response_bn'] = df.response_bn.fillna('').astype(str)

def band_of(row):
    if 'ভাবার্থ' in row.prompt_bn or 'শাব্দিক অর্থ' in row.prompt_bn: return 'idiom'
    if str(row.context) != '[NULL]': return 'ctx'
    return 'gk'

samp_p, samp_s = predict_df(samp, R)
test_p, test_s = predict_df(test, R)
samp_band = samp.apply(band_of, axis=1).values
test_band = test.apply(band_of, axis=1).values
samp_res = np.isin(samp_s, ['residual', 'ctx'])
test_res = np.isin(test_s, ['residual', 'ctx'])

samp_fz = fuzzy_mmlu_pass(samp, samp_res, mm_bank)
test_fz = fuzzy_mmlu_pass(test, test_res, mm_bank)
test_sb = sample_bank_pass(test, test_res, samp)
y = samp.label.values
fzi = np.array(sorted(samp_fz)); fzp = np.array([samp_fz[i][0] for i in fzi])
print(f'sample fuzzy hits {len(samp_fz)} acc={(fzp==y[fzi]).mean():.3f} | test fuzzy {len(test_fz)} sample-bank {len(test_sb)}')

undecided_t = [i for i in np.where(test_res)[0] if i not in test_fz and i not in test_sb]
undecided_s = [i for i in np.where(samp_res)[0] if i not in samp_fz]
print('undecided: test', len(undecided_t), 'sample', len(undecided_s))
print(pd.Series(test_band[undecided_t]).value_counts().to_dict())


## Offline Wikipedia evidence cache (gk verify+answer; graceful if absent)

In [ ]:
# ---- offline evidence: wiki cache (old + cleaned keys) ----
WIKI_CACHE = {}
p = os.path.join(SRC_DIR, 'wiki_pages_cache.json')
if os.path.exists(p):
    WIKI_CACHE = json.load(open(p, encoding='utf-8'))
print('wiki cache entries:', len(WIKI_CACHE))

def clean_query(question):
    q = str(question)
    q = re.split(r'[\?।]', q)[0]
    q = re.sub(r'[কখগঘঙ]\)\s*[^,;]+[,;]?', ' ', q)
    q = re.sub(r'_{2,}|-{3,}', ' ', q)
    return re.sub(r'\s+', ' ', q).strip()

def wiki_pages(question, npages=3):
    for key in (clean_query(question), re.sub(r'[?।]', ' ', str(question)).strip()):
        if key in WIKI_CACHE:
            return WIKI_CACHE[key]
    if ON_KAGGLE:   # phase-1 fallback: live fetch (internet ON)
        try:
            import requests
            r = requests.get('https://bn.wikipedia.org/w/api.php', params={
                'action': 'query', 'generator': 'search', 'gsrsearch': clean_query(question),
                'gsrlimit': npages, 'prop': 'extracts', 'explaintext': 1, 'format': 'json'},
                headers={'User-Agent': 'datathon-research/1.0'}, timeout=30)
            pages = r.json().get('query', {}).get('pages', {})
            out = [{'title': p_.get('title', ''), 'text': p_.get('extract', '')}
                   for p_ in sorted(pages.values(), key=lambda p_: p_.get('index', 9))]
            WIKI_CACHE[clean_query(question)] = out
            return out
        except Exception:
            return []
    return []

def toks(s): return set(w for w in norm_sp(s).split() if len(w) > 1)

def evidence_sentences(question, response, max_sents=12):
    qt = toks(question) | toks(response)
    scored = []
    for p_ in wiki_pages(question):
        for s in re.split(r'(?<=[।\n])\s*', p_['text']):
            if len(s.strip()) < 8: continue
            st = toks(s)
            if st:
                sc = len(st & qt) / (len(qt) ** 0.5)
                if sc > 0: scored.append((sc, p_['title'], s.strip()))
    scored.sort(key=lambda x: -x[0])
    return [f'[{t}] {s}' for _, t, s in scored[:max_sents]]

# idiom dictionaries
IDIOM_AUX = {}
for fn in ['eb_bagdhara.json', 'wiktionary_defs.json']:
    p = os.path.join(SRC_DIR, fn)
    if os.path.exists(p):
        d = json.load(open(p, encoding='utf-8'))
        for k, v in d.items():
            IDIOM_AUX.setdefault(norm_sp(k), []).extend(v if isinstance(v, list) else [v])
print('aux idiom entries:', len(IDIOM_AUX))


## Apply CPU passes (idiom · math · grammar · antonym); collect rows needing an LLM

In [ ]:
from rapidfuzz import fuzz as _f
final = test_p.copy()
for i in test_sb: final[i] = test_sb[i][0]   # sample-bank first...
for i in test_fz: final[i] = test_fz[i][0]   # ...fuzzy WINS on conflict (matches v10/v17)

def is_idiom(p): return 'ভাবার্থ' in str(p) or 'শাব্দিক অর্থ' in str(p) or 'আক্ষরিক' in str(p)
def is_proverb(p): return ('প্রবাদ' in str(p) or 'বাগধারা' in str(p)) and not is_idiom(p)
def extract_q(p):
    m=re.search(r'[\'‘’"“”]([^\'‘’"“”]+)[\'‘’"“”]', str(p)); return m.group(1) if m else None
def idiom_meanings(prompt):
    t=extract_q(prompt)
    if not t: return None
    k=norm_sp(t); e=LEX.get(k) or (LEX.get(ALIASES[k]) if k in ALIASES else None)
    if not e: return None
    lit = 'শাব্দিক' in str(prompt) or 'আক্ষরিক' in str(prompt)
    return (e['lit'] if lit else e['fig']) or e['fig'] or e['lit'] or None

ctx_rows, prov_rows, gk_rows = [], [], []
for i in np.where(test_res)[0]:
    i=int(i)
    if i in test_fz or i in test_sb: continue
    r = test.iloc[i]; p = r.prompt_bn
    gv, _ = gr.predict(p, r.context, r.response_bn)
    if gv is not None: final[i] = gv; continue
    if is_idiom(p): continue   # handled by the dedicated il.predict idiom loop below (exact v13 logic)
    mv = mt.predict(p, r.response_bn)                 # v18-v26 deterministic math (LCM+speed misfires fixed 07-10)
    if mv is not None: final[i] = mv; continue
    cfv = cf.predict(p, r.response_bn)
    if cfv is not None: final[i] = cfv; continue
    sdv = sd.predict(p, r.response_bn)
    if sdv is not None: final[i] = sdv; continue
    abv = ab.predict(p, r.response_bn)
    if abv is not None: final[i] = abv; continue
    lmv = lm.predict(p, r.context, r.response_bn)
    if lmv is not None: final[i] = lmv; continue
    wbv = wb.predict(p, r.response_bn)
    if wbv is not None: final[i] = wbv; continue
    # --- 2026-07-10 CENSUS SOLVERS (deterministic, abstain-safe; the v27->v33 climb 0.909->0.922) ---
    alv = al.predict(p, r.response_bn)               # linear/quadratic equations
    if alv is not None: final[i] = alv; continue
    wdv = wd.predict(p, r.response_bn)               # day-of-week arithmetic
    if wdv is not None: final[i] = wdv; continue
    msv = ms.predict(p, r.response_bn)               # relative motion (meeting-time / separation)
    if msv is not None: final[i] = msv; continue
    pzv = pz.predict(p, r.response_bn)               # sequence/percent/word-perm/binary/quad-ineq/subst/log/mul-puzzle/trig/geo
    if pzv is not None: final[i] = pzv; continue
    # --- end census solvers ---
    if 'বিপরীত' in str(p):
        w=re.search(r'[\'‘’"“”]?([^\s\'‘’"“”]+)[\'‘’"“”]?\s*শব্দের', str(p))
        a = ANT.get(norm_sp(w.group(1))) if w else None
        if a: final[i] = 1 if max(_f.ratio(norm(r.response_bn),norm(x)) for x in a)>=80 else 0; continue
        ctx_rows.append(i); continue
    if is_proverb(p) and idiom_meanings(p): prov_rows.append(i); continue
    if str(r.context) != '[NULL]': ctx_rows.append(i)
    else: gk_rows.append(i)
# idiom band: il.predict on ALL idiom rows (exact v13 logic — covers non-residual idiom rows the loop skips)
for i in range(len(test)):
    if is_idiom(test.prompt_bn.iloc[i]):
        v = il.predict(test.prompt_bn.iloc[i], test.response_bn.iloc[i], LEX)
        if v is not None: final[i] = v
VERD = json.load(open('_faithful_verdicts.json',encoding='utf-8')) if os.path.exists('_faithful_verdicts.json') else {}
def savev(): json.dump(VERD, open('_faithful_verdicts.json','w',encoding='utf-8'), ensure_ascii=False)
print(f'LLM rows -> ctx {len(ctx_rows)} | proverb {len(prov_rows)} | gk {len(gk_rows)}')

In [ ]:
if bett:
    
    # ── ctx band: HF Qwen3-32B judge ──
    import torch, time, gc
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    def flush_gpu(): gc.collect(); torch.cuda.empty_cache()
    SYS_J = 'You are a meticulous fact-checker for Bengali (Bangladesh). Beware plausible globally-famous substitutions for Bangladesh-specific facts. Answer exactly one word: Yes or No.'
    todo_ctx = [i for i in ctx_rows if ('ctx:'+str(test.id.iloc[i])) not in VERD]
    print(len(todo_ctx),'ctx rows for Qwen3-32B')
    if todo_ctx:
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        mmx={i:'13GiB' for i in range(torch.cuda.device_count())}
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto', quantization_config=quant, max_memory=mmx); m.eval()
        def qgen(sysm, user, max_new=8):
            txt = tok.apply_chat_template([{'role':'system','content':sysm},{'role':'user','content':user}], tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for ml in (3072, 1536):
                try:
                    enc = tok(txt, return_tensors='pt', truncation=True, max_length=ml).to(m.device)
                    with torch.no_grad(): out = m.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id)
                    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
                except torch.cuda.OutOfMemoryError: torch.cuda.empty_cache()
            return ''
        t0=time.time()
        for n,i in enumerate(todo_ctx):
            r=test.iloc[i]
            u=f'Passage (may be incomplete):\n{r.context}\n\nQuestion (Bengali): {r.prompt_bn}\nCandidate answer: {r.response_bn}\n\nUsing the passage AND your knowledge, is the candidate answer factually correct and a valid answer? Yes or No.'
            v=1 if 'yes' in qgen(SYS_J,u).lower()[:12] else 0
            VERD['ctx:'+str(r.id)]={'v':int(v)}; final[i]=v
            if n%20==0: savev(); torch.cuda.empty_cache(); rate=(n+1)/(time.time()-t0); print(f'  {n}/{len(todo_ctx)}  {rate:.2f}/s',flush=True)
        savev(); del m, tok; flush_gpu()
    for i in ctx_rows:
        rid='ctx:'+str(test.id.iloc[i])
        if rid in VERD: final[i]=VERD[rid]['v']
    print('ctx band done'); [print(f'GPU{g} free {torch.cuda.mem_get_info(g)[0]/1e9:.1f}GB') for g in range(torch.cuda.device_count())]
    
    # ── Start Ollama ──
    import subprocess, time, requests, shutil
    subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True, check=False)
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=False)
    OLLAMA_BIN = shutil.which('ollama') or '/usr/local/bin/ollama'
    assert os.path.exists(OLLAMA_BIN) or shutil.which('ollama'), 'ollama binary not found after install'
    os.environ.setdefault('OLLAMA_HOST','127.0.0.1:11434')
    subprocess.Popen([OLLAMA_BIN,'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try: requests.get('http://localhost:11434/api/tags', timeout=2); print('ollama up'); break
        except Exception: time.sleep(1)
    for mdl in [TIGER_GGUF, GEMMA]:
        print('pull', mdl, flush=True); subprocess.run([OLLAMA_BIN,'pull',mdl], check=False)
    def ollama_chat(model, sysm, user, num_predict, num_ctx, think=None):
        p={'model':model,'messages':[{'role':'system','content':sysm},{'role':'user','content':user}],'stream':False,'options':{'temperature':0,'num_predict':num_predict,'num_ctx':num_ctx}}
        if think is not None: p['think']=think
        for _ in range(3):
            try: return requests.post('http://localhost:11434/api/chat', json=p, timeout=300).json()['message']['content'].strip()
            except Exception: time.sleep(2)
        return ''
    print('ollama ready')
    
    # ── proverb + gk (unchanged v17 logic) ──
    GK_TH = 60
    SYS_P = ('You compare two meanings of a Bengali idiom/proverb. You are given the idiom, its CORRECT '
             'meaning, and a CANDIDATE meaning. Decide if the candidate expresses the SAME figurative sense '
             'as the correct meaning (paraphrases count as same; opposite or unrelated senses count as '
             'different). Answer EXACTLY one word: Yes or No.')
    todo_p = [i for i in prov_rows if ('prov:'+str(test.id.iloc[i])) not in VERD]
    print(len(todo_p),'proverb rows (Ollama Gemma)')
    for n,i in enumerate(todo_p):
        r=test.iloc[i]; mn=idiom_meanings(r.prompt_bn); correct='; '.join(mn[:4])
        u=f'Idiom: {extract_q(r.prompt_bn)}\nCorrect meaning: {correct}\nCandidate meaning: {r.response_bn}\n\nDoes the candidate express the same figurative sense as the correct meaning? Yes or No.'
        t=ollama_chat(GEMMA, SYS_P, u, 6, 2048).lower()
        v = 1 if ('yes' in t[:8] or '\u09b9\u09cd\u09af\u09be\u0981' in t[:8]) else 0
        VERD['prov:'+str(r.id)]={'v':int(v)}; final[i]=v
        if n%20==0: savev()
    for i in prov_rows:
        rid='prov:'+str(test.id.iloc[i])
        if rid in VERD: final[i]=VERD[rid]['v']
    savev(); print('proverb done')
    
    SYS_V = ('You are a strict fact-checker for Bengali (Bangladesh) knowledge. Given a question and a '
             'candidate answer, decide if the candidate answer is factually correct. Answer EXACTLY one word: Yes or No.')
    SYS_A = 'You answer Bengali factual questions with the shortest correct answer. No explanations.'
    todo_gk = [i for i in gk_rows if ('gk:'+str(test.id.iloc[i])) not in VERD]
    print(len(todo_gk),'gk rows (Ollama TigerLLM verify + Gemma answer)')
    t0=time.time()
    for n,i in enumerate(todo_gk):
        r=test.iloc[i]; ev=evidence_sentences(r.prompt_bn, r.response_bn)
        evb='\n'.join(ev[:8]) if ev else '(no evidence)'
        uv=f'Evidence:\n{evb}\n\nQuestion: {r.prompt_bn}\nCandidate answer: {r.response_bn}\n\nIs the candidate answer factually correct? Yes or No.'
        tv=ollama_chat(TIGER_GGUF, SYS_V, uv, 6, 4096).lower()
        verify = 1 if ('yes' in tv[:10] or 'হ্যাঁ' in tv[:10] or 'হা' in tv[:6]) else 0
        evk='\n'.join(ev) if ev else '(no evidence found)'
        ua=(f'Evidence sentences from Bengali Wikipedia:\n{evk}\n\nQuestion (Bengali): {r.prompt_bn}\n\n'
            'Answer the question in Bengali as briefly as possible (a name, date, number, or short phrase). '
            'Prefer the evidence; if the evidence does not contain the answer, use your own knowledge. '
            'If you truly cannot answer, reply exactly: অজানা')
        ans=ollama_chat(GEMMA, SYS_A, ua, 64, 8192, think=False)
        acomp = 1 if sim('' if 'অজানা' in ans else ans, r.response_bn) >= GK_TH else 0
        v = 1 if (verify or acomp) else 0
        VERD['gk:'+str(r.id)]={'v':int(v)}; final[i]=v
        if n%20==0: savev(); rate=(n+1)/(time.time()-t0); print(f'  {n}/{len(todo_gk)}  {rate:.2f}/s  ETA {(len(todo_gk)-n)/max(rate,1e-6)/60:.0f}m',flush=True)
    for i in gk_rows:
        rid='gk:'+str(test.id.iloc[i])
        if rid in VERD: final[i]=VERD[rid]['v']
    savev(); print('gk band done')
else:
    
    import time, gc
    
    GK_TH  = 60
    CTX_TH = 60
    
    # ── Initialize RAGStore (DO NOT call load_all) ──
    _SEARCH = [p for p in [SRC_DIR, CODE_DIR, '/kaggle/input', '.'] if p and os.path.exists(p)]
    rag_store = RAGStore()
    
    rag_store.add(Services2RAG(_SEARCH))
    rag_store.add(KnowledgeBaseRAG(_SEARCH))
    rag_store.add(LawRAG(_SEARCH))
    rag_store.add(BookRAG(_SEARCH))
    rag_store.add(SomashRAG(_SEARCH))
    rag_store.add(ServicesRAG(_SEARCH))
    rag_store.add(SattRAG(_SEARCH))
    rag_store.add(EBanglaRAG(_SEARCH))
    
    # ── Gather RAG evidence for ctx_rows + gk_rows (CPU, backend-by-backend) ──
    # ── PURGE CPU-PHASE VARIABLES BEFORE RAG ──
    try:
        # Delete heavy variables that are no longer needed
        del mm_bank, samp, samp_p, samp_s, samp_band, samp_fz, test_fz, test_sb
        del R  # Delete the Resources object if it holds large dictionaries
    except Exception:
        pass

    import gc, ctypes
    gc.collect()
    gc.collect()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass
        
    import psutil
    print(f"-> RAM after pre-RAG purge: {psutil.virtual_memory().used / (1024**3):.1f}GB", flush=True)

    # ── Gather RAG evidence for ctx_rows + gk_rows (CPU, backend-by-backend) ──
    _ctx_evidence = {}
    _gk_evidence = {}
    _ctx_evidence = {}
    _gk_evidence = {}
    
    if ctx_rows or gk_rows:
        # Initialize evidence lists for accumulating text chunks across all backends
        for i in ctx_rows: _ctx_evidence[i] = {'ev_dicts': []}
        for i in gk_rows:  _gk_evidence[i] = {'ev_dicts': []}
        
        # Process each backend ONE AT A TIME to minimize RAM footprint
        for backend in rag_store.backends:
            print(f"\n[{backend.name}] ---------------------------------")
            import torch, psutil
            ram_before = psutil.virtual_memory().used / (1024**3)
            vram_before = torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0
            print(f"-> Starting RAM: {ram_before:.1f}GB | VRAM: {vram_before:.1f}GB", flush=True)
            
            print(f"-> Loading RAG backend: {backend.name}...", flush=True)
            backend.load()
            
            # Gather sentences from this specific index for ctx_rows
            for i in ctx_rows:
                r = test.iloc[i]
                try:
                    chunks = backend.retrieve_facts(clean_query(r.prompt_bn), top_k=5)
                    qt = toks(r.prompt_bn) | toks(r.response_bn)
                    for chunk in chunks:
                        for s in re.split(r'(?<=[।\n])\s*', chunk['text']):
                            if len(s.strip()) < 8: continue
                            st = toks(s)
                            if st:
                                sc = len(st & qt) / (len(qt) ** 0.5)
                                if sc > 0:
                                    _ctx_evidence[i]['ev_dicts'].append({'text': s.strip(), 'score': sc})
                except Exception:
                    pass
            
            # Gather sentences from this specific index for gk_rows
            for i in gk_rows:
                r = test.iloc[i]
                try:
                    chunks = backend.retrieve_facts(clean_query(r.prompt_bn), top_k=5)
                    qt = toks(r.prompt_bn) | toks(r.response_bn)
                    for chunk in chunks:
                        for s in re.split(r'(?<=[।\n])\s*', chunk['text']):
                            if len(s.strip()) < 8: continue
                            st = toks(s)
                            if st:
                                sc = len(st & qt) / (len(qt) ** 0.5)
                                if sc > 0:
                                    _gk_evidence[i]['ev_dicts'].append({'text': s.strip(), 'score': sc})
                except Exception:
                    pass
            
            # Force garbage collection immediately after dropping the backend index
            # Force garbage collection immediately after dropping the backend index
            backend.free()
            
            import gc
            import ctypes
            
            # 1. Standard Python garbage collection
            gc.collect()
            gc.collect()
            
            # 2. Force Linux to reclaim the memory pages instantly
            try:
                ctypes.CDLL('libc.so.6').malloc_trim(0)
            except Exception:
                pass

            import torch, psutil
            ram_after = psutil.virtual_memory().used / (1024**3)
            vram_after = torch.cuda.memory_allocated() / (1024**3) if torch.cuda.is_available() else 0
            print(f"-> Freed {backend.name}. RAM: {ram_after:.1f}GB | VRAM: {vram_after:.1f}GB", flush=True)
            
        # Post-process: Sort and select the top 12 sentences globally (identical to original logic)
        for i in ctx_rows:
            r = test.iloc[i]
            _ctx_evidence[i]['ev_dicts'].sort(key=lambda x: -x['score'])
            ev = [d['text'] for d in _ctx_evidence[i]['ev_dicts'][:12]]
            ctx_text = str(r.context).strip()
            _ctx_evidence[i]['combined'] = ctx_text + '\n' + '\n'.join(ev[:8]) if ev else ctx_text
            _ctx_evidence[i]['ev'] = ev
            
        for i in gk_rows:
            _gk_evidence[i]['ev_dicts'].sort(key=lambda x: -x['score'])
            _gk_evidence[i]['ev'] = [d['text'] for d in _gk_evidence[i]['ev_dicts'][:12]]

        del rag_store
        gc.collect()
        print(f'RAG evidence gathered safely: ctx={len(_ctx_evidence)}, gk={len(_gk_evidence)}')
   # ── Start Ollama (Multi-GPU & Concurrency Configured) ──
    import subprocess, requests, shutil, time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    subprocess.run('apt-get update -qq && apt-get install -y -qq zstd', shell=True, check=False, capture_output=True)
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=False, capture_output=True)
    OLLAMA_BIN = shutil.which('ollama') or '/usr/local/bin/ollama'
    assert os.path.exists(OLLAMA_BIN) or shutil.which('ollama'), 'ollama binary not found'
    
    os.environ.setdefault('OLLAMA_HOST','127.0.0.1:11434')
    os.environ.setdefault('OLLAMA_MODELS', '/kaggle/working/ollama_models')
    
    # 1. Force Ollama to use both GPUs for up to 4 simultaneous queries
    os.environ['OLLAMA_NUM_PARALLEL'] = '4' 
    os.environ['OLLAMA_MAX_VRAM'] = '30000000000'
    
    subprocess.Popen([OLLAMA_BIN,'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try: requests.get('http://localhost:11434/api/tags', timeout=2); print('ollama up'); break
        except Exception: time.sleep(1)
        
    for mdl in [TIGER_GGUF, GEMMA]:
        print('pull', mdl, flush=True)
        subprocess.run([OLLAMA_BIN,'pull',mdl], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
    def ollama_chat(model, sysm, user, num_predict, num_ctx, think=None):
        # Keep alive for 5 minutes during the active loop
        p={'model':model,'messages':[{'role':'system','content':sysm},{'role':'user','content':user}],'stream':False,'keep_alive':'5m','options':{'temperature':0,'num_predict':num_predict,'num_ctx':num_ctx}}
        if think is not None: p['think']=think
        for _ in range(3):
            try: return requests.post('http://localhost:11434/api/chat', json=p, timeout=300).json()['message']['content'].strip()
            except Exception: time.sleep(2)
        return ''

    def unload_model(model_name):
        """Forces Ollama to immediately evict the model from all GPUs."""
        print(f"\n-> Force unloading {model_name} from VRAM...", flush=True)
        try:
            requests.post('http://localhost:11434/api/chat', json={'model': model_name, 'keep_alive': 0}, timeout=5)
        except Exception:
            pass
            
    print('ollama ready')
    
    # ── System prompts ──
    SYS_V = ('You are a strict fact-checker for Bengali (Bangladesh) knowledge. Given a question and a '
             'candidate answer, decide if the candidate answer is factually correct. Answer EXACTLY one word: Yes or No.')
    SYS_A = 'You answer Bengali factual questions with the shortest correct answer. No explanations.'
    SYS_P = ('You compare two meanings of a Bengali idiom/proverb. You are given the idiom, its CORRECT '
             'meaning, and a CANDIDATE meaning. Decide if the candidate expresses the SAME figurative sense '
             'as the correct meaning (paraphrases count as same; opposite or unrelated senses count as '
             'different). Answer EXACTLY one word: Yes or No.')
    
    # ── PASS 1: ALL TigerLLM calls (ctx + gk) ──
    _tiger_results = {}
    tiger_queue = []
    
    for i in ctx_rows:
        r = test.iloc[i]
        evb = _ctx_evidence[i]['combined'] if i in _ctx_evidence else str(r.context).strip()
        if not evb: evb = '(no evidence)'
        uv = f'Evidence:\n{evb}\n\nQuestion: {r.prompt_bn}\nCandidate answer: {r.response_bn}\n\nIs the candidate answer factually correct? Yes or No.'
        tiger_queue.append((i, uv))
    
    for i in gk_rows:
        r = test.iloc[i]
        ev = _gk_evidence[i]['ev'] if i in _gk_evidence else []
        evb = '\n'.join(ev[:8]) if ev else '(no evidence)'
        uv = f'Evidence:\n{evb}\n\nQuestion: {r.prompt_bn}\nCandidate answer: {r.response_bn}\n\nIs the candidate answer factually correct? Yes or No.'
        tiger_queue.append((i, uv))
    
    print(f'TigerLLM pass: {len(tiger_queue)} rows')
    
    def process_tiger(item):
        i, uv = item
        if ('ctx:'+str(test.id.iloc[i])) in VERD or ('gk:'+str(test.id.iloc[i])) in VERD:
            rid = 'ctx:'+str(test.id.iloc[i]) if i in dict.fromkeys(ctx_rows) else 'gk:'+str(test.id.iloc[i])
            if rid in VERD: return i, VERD[rid]['v']
        tv = ollama_chat(TIGER_GGUF, SYS_V, uv, 6, 4096).lower()
        return i, 1 if ('yes' in tv[:10] or 'হ্যাঁ' in tv[:10] or 'হা' in tv[:6]) else 0

    t0 = time.time()
    # 2. Process concurrently across both GPUs
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(process_tiger, item): item for item in tiger_queue}
        for n, future in enumerate(as_completed(futures)):
            i, verify = future.result()
            _tiger_results[i] = {'verify': verify}
            if (n+1) % 20 == 0:
                print(f'  Tiger {n+1}/{len(tiger_queue)}  {(n+1)/(time.time()-t0):.2f}/s', flush=True)

    # 3. Destroy TigerLLM from VRAM before moving to Gemma
    unload_model(TIGER_GGUF)
    
    # ── PASS 2: ALL Gemma calls (ctx + prov + gk) ──
    _gemma_results = {}
    gemma_queue = []
    
    for i in ctx_rows:
        r = test.iloc[i]
        evb = _ctx_evidence[i]['combined'] if i in _ctx_evidence else str(r.context).strip()
        if not evb: evb = '(no evidence)'
        ua = (f'Evidence sentences from the given passage:\n{evb}\n\n'
              f'Question (Bengali): {r.prompt_bn}\n\n'
              'Answer the question in Bengali as briefly as possible '
              '(a name, date, number, or short phrase). '
              'Prefer the evidence; if the evidence does not contain the answer, '
              'use your own knowledge. '
              'If you truly cannot answer, reply exactly: অজানা')
        gemma_queue.append((i, ua, SYS_A, 64, 8192, 'ctx'))
    
    for i in prov_rows:
        r = test.iloc[i]; mn = idiom_meanings(r.prompt_bn); correct = '; '.join(mn[:4])
        u = (f'Idiom: {extract_q(r.prompt_bn)}\n'
             f'Correct meaning: {correct}\n'
             f'Candidate meaning: {r.response_bn}\n\n'
             'Does the candidate express the same figurative sense as the correct meaning? Yes or No.')
        gemma_queue.append((i, u, SYS_P, 6, 2048, 'prov'))
    
    for i in gk_rows:
        r = test.iloc[i]
        ev = _gk_evidence[i]['ev'] if i in _gk_evidence else []
        evk = '\n'.join(ev) if ev else '(no evidence found)'
        ua = (f'Evidence sentences from Bengali Wikipedia:\n{evk}\n\n'
              f'Question (Bengali): {r.prompt_bn}\n\n'
              'Answer the question in Bengali as briefly as possible '
              '(a name, date, number, or short phrase). '
              'Prefer the evidence; if the evidence does not contain the answer, '
              'use your own knowledge. '
              'If you truly cannot answer, reply exactly: অজানা')
        gemma_queue.append((i, ua, SYS_A, 64, 8192, 'gk'))
    
    print(f'Gemma pass: {len(gemma_queue)} rows')
    
    def process_gemma(item):
        i, prompt, sys_prompt, max_tok, ctx_size, band = item
        ans = ollama_chat(GEMMA, sys_prompt, prompt, max_tok, ctx_size, think=False)
        return i, ans, band

    t0 = time.time()
    # 4. Process concurrently across both GPUs
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(process_gemma, item): item for item in gemma_queue}
        for n, future in enumerate(as_completed(futures)):
            i, ans, band = future.result()
            _gemma_results[i] = {'raw': ans, 'band': band}
            if (n+1) % 20 == 0:
                print(f'  Gemma {n+1}/{len(gemma_queue)}  {(n+1)/(time.time()-t0):.2f}/s', flush=True)
                
    # 5. Destroy Gemma from VRAM
    unload_model(GEMMA)
    # ── PASS 3: Merge results ──
    for i in ctx_rows:
        r = test.iloc[i]
        verify = _tiger_results[i]['verify']
        ans = _gemma_results[i]['raw']
        acomp = 1 if sim('' if 'অজানা' in ans else ans, r.response_bn) >= CTX_TH else 0
        v = 1 if (verify or acomp) else 0
        VERD['ctx:'+str(r.id)] = {'v': int(v)}; final[i] = v
    
    for i in prov_rows:
        r = test.iloc[i]
        t = _gemma_results[i]['raw'].lower()
        v = 1 if ('yes' in t[:8] or 'অজানা' in t[:8]) else 0
        VERD['prov:'+str(r.id)] = {'v': int(v)}; final[i] = v
    
    for i in gk_rows:
        r = test.iloc[i]
        verify = _tiger_results[i]['verify']
        ans = _gemma_results[i]['raw']
        acomp = 1 if sim('' if 'অজানা' in ans else ans, r.response_bn) >= GK_TH else 0
        v = 1 if (verify or acomp) else 0
        VERD['gk:'+str(r.id)] = {'v': int(v)}; final[i] = v
    
    savev()
    print('All bands done (RAG-based, no Qwen)')

## Write submission

In [ ]:
sub = pd.DataFrame({'id': test.id, 'label': [int(x) for x in final]})
out = '/kaggle/working/submission.csv' if ON_KAGGLE else 'submission_v17_faithful.csv'
sub.to_csv(out, index=False)
assert len(sub)==len(test) and set(sub.label.unique())<={0,1}
print('WROTE', out, '| rows', len(sub), '| dist', sub.label.value_counts().to_dict())
